In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from selenium import webdriver

from bs4 import BeautifulSoup

import pandas as pd

from time import sleep

import datetime

from pandas import ExcelWriter
import re
import os
from seleniumbase import SB
from seleniumbase.config import settings


In [3]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'GT SIB' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.10.0")

now=datetime.datetime.now()

filename= '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

# scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')



tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)






Running GT SIB Web Scraping Tool v.1.10.0


In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty

    return sqldict


In [5]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}   


regdict={
# 'GT SIB 1': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=251',# more one column, change a little logic
# 'GT SIB 2': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=252',
# 'GT SIB 3': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=253',
'GT SIB 4': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=254',
# 'GT SIB 5': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=255',
# 'GT SIB 6': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=257', 
# 'GT SIB 7': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=258',
# 'GT SIB 8': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=259',
# 'GT SIB 9': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=260',
# 'GT SIB 10': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=452',

    }


Typology={

        regulatorName+' 1': 'INSTITUCIONES BANCARIAS',
        regulatorName+' 2': 'SOCIEDADES FINANCIERAS',
        regulatorName+' 3': 'COMPANIAS DE ALMACENADORAS',
        regulatorName+' 4': 'COMPANIAS DE SEGUROS',
        regulatorName+' 5': 'CASAS DE CAMBIO',
        regulatorName+' 6': 'CASAS DE BOLSA',
        regulatorName+' 7': 'TARJETAS DE CREDITO',
        regulatorName+' 8': 'OTRAS INSTITUCIONES',
        regulatorName+' 9': 'GRUPOS FINANCIEROS',
        regulatorName+' 10': 'INSTITUCIONES DE MICROFINANZAS',

        }
processdate = now.strftime('%Y-%m-%d')




In [6]:
#------------------------------------------------ Begin_Main----------------------------------------
# def verify_success(sb):
#     try:
#         sb.assert_element('img[alt="Logo Assembly"]', timeout=4)
#         sb.sleep(3)

#     except Exception as e:
#         print(f"Verification failed: {e}")


settings.DOWNLOADS_FOLDER = tempfolder  
settings.DISABLE_LOGS = True 

for reg in regdict:
    print('Working: ', reg)
    with SB(uc=True) as sb:
        sb.uc_open_with_reconnect(regdict[reg])

        
        

        sleep(10)
        page_source = sb.get_page_source()
        soup = BeautifulSoup(page_source, "html.parser")
        first_table = soup.find("vaadin-vertical-layout")
        colunm_length = len(soup.find_all('vaadin-grid-column'))
        if colunm_length is None:
                snippet = soup.prettify()[:2000]  # keep the traceback readable
                raise RuntimeError(f"vaadin-grid missing; soup snapshot:\n{snippet}")

        
        if reg == 'GT SIB 1':
            field_map = {
                1: "Name",
                2: "Address_1",
                3: "Phone",
                4: "Fax",
                5: None,            
                6: "Website",
                7: "RegulationDate",
            }
        else:
            field_map = {
                1: "Name",
                2: "Address_1",
                3: "Phone",
                4: "Fax",          
                5: "Website",
                6: "RegulationDate",
            }            
        
        current_row = None
        records = []    

        for index, cell in enumerate(soup.select("vaadin-grid-cell-content[slot]")):
            slot = int(cell["slot"].split("-")[-1])
            text_value = cell.get_text(strip=True)

            if not text_value:
                continue

            slot_mod = slot % colunm_length
            if slot_mod == 0:
                continue  # skip the column you’re intentionally ignoring

            if slot_mod == 1:
                if current_row:
                    records.append(current_row)
                current_row = {field: "" for field in field_map.values() if field}
                current_row["ListProcessDate"] = processdate
                current_row["RegulationType"] = "Regulated"
                current_row["RegCtry"] = reg.split()[0]
                current_row["RegCode"] = reg.split()[1]
                current_row["ListCode"] = reg.split()[-1]
                current_row["ListName"] = Typology[reg]
            fld = field_map.get(slot_mod)
            print(f"{fld}: {text_value}")
            if fld:
                current_row[fld] = text_value
        # grab the final row
        if current_row:
            records.append(current_row)
            
        for row in records:
            for key in ["Name","Address_1","Phone","Fax","Website","RegulationDate",
                        "ListProcessDate","RegulationType","RegCtry","RegCode",
                        "ListCode","ListName"]:
                sqldict[key].append(row.get(key))   
        sqldict = bourange_same_length_array(sqldict)
        sb.sleep(2)
            


Working:  GT SIB 4
Name: DEPARTAMENTO DE SEGUROS Y PREVISIÓN DE EL CRÉDITO HIPOTECARIO NACIONAL DE GUATEMALA
Address_1: Avenida Reforma 6-64, Zona 9, Plaza Corporativa Reforma, Torre I, Nivel 1 y 5
Phone: 2290-7400
Fax: 2339-0843
Website: www.chn.com.gt
Name: SEGUROS G&T, S. A.
Address_1: Ruta 2 2-39, Zona 4 (Notificar en Edificio I, Nivel 5, Puerta 2)
Phone: 2338-5858
Fax: 2332-9081/2331-3270
Website: www.segurosgyt.com.gt
Name: BMI COMPAÑÍA DE SEGUROS DE GUATEMALA, S. A.
Address_1: 15 calle 1-11 zona 10, Edificio TerraEsperanza, Nivel 3 Oficina 301
Phone: 25012222
Fax: --
Website: www.bmi.gt
Name: SEGUROS UNIVERSALES, S. A.
Address_1: 4a. Calle 7-73, Zona 9
Phone: 1789/2384-7400/2384-7500
Fax: 2332-3372
Website: www.segurosuniversales.com
Name: ASSA COMPAÑÍA DE SEGUROS, S. A.
Address_1: 10a. Avenida 14-14 Zona 14,Edificio ASSA
Phone: 2356-2700
Fax: N/A
Website: www.assanet.com
Name: PAN-AMERICAN LIFE INSURANCE DE GUATEMALA, COMPAÑÍA DE SEGUROS, S. A.
Address_1: Avenida La Reforma 9-0

In [8]:
from seleniumbase import SB
from bs4 import BeautifulSoup
url = "https://www.sib.gob.gt/ConsultaDinamica/?cons=254"
with SB(uc=True) as sb:
    sb.uc_open_with_reconnect(url)
    sb.sleep(5)  # wait for Vaadin to render

    html = sb.execute_script(
        """
        const grid = document.querySelector('vaadin-grid');
        if (!grid || !grid.shadowRoot) { return ''; }
        const scroller = grid.shadowRoot
            .querySelector('vaadin-grid-table')
            ?.shadowRoot?.querySelector('#scroller');
        return scroller ? scroller.innerHTML : '';
        """
    )

    soup = BeautifulSoup(html, "html.parser")
    for row in soup.select("tr"):
        cells = [td.get_text(strip=True) for td in row.select("td")]
        print(cells)


In [13]:
from seleniumbase import SB

def fetch_grid_rows(sb):
    return sb.execute_script(
        """
        const grid = document
            .querySelector('body > vaadin-vertical-layout')
            ?.shadowRoot
            ?.querySelector('vaadin-vertical-layout vaadin-grid');
        if (!grid) { return []; }

        const scroller = grid.shadowRoot
            ?.querySelector('vaadin-grid-table')
            ?.shadowRoot
            ?.querySelector('#scroller');
        if (!scroller) { return []; }

        return Array.from(scroller.querySelectorAll('tr'), row =>
            Array.from(row.querySelectorAll('td slot'), slot =>
                slot.assignedNodes()
                    .map(n => n.textContent.trim())
                    .filter(Boolean)
                    .join(' ')
            )
        );
        """
    )

with SB(uc=True) as sb:
    sb.uc_open_with_reconnect("https://www.sib.gob.gt/ConsultaDinamica/?cons=254")
    sb.sleep(5)  # wait for Vaadin to render the grid
    rows = fetch_grid_rows(sb)
    for row in rows:
        print(row)


In [18]:

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)

mask = df["Phone"].apply(
    lambda x: bool(re.match(r"\d", str(x).strip())) if pd.notna(x) else False
)
df = df[mask].copy()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
sleep(3)

driver.quit()

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_27520\3140535103.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [19]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 0 values.
Key 'priority' has 0 values.
Key 'ListLabel' has 0 values.
Key 'Typology' has 0 values.
Key 'EntryType' has 0 values.
Key 'Name' has 0 values.
Key 'InternalID_1' has 0 values.
Key 'InternalID_1_type' has 0 values.
Key 'InternalID_2' has 0 values.
Key 'InternalID_2_type' has 0 values.
Key 'InternalID_3' has 0 values.
Key 'InternalID_3_type' has 0 values.
Key 'CoType' has 0 values.
Key 'License_Type' has 0 values.
Key 'Address_1' has 0 values.
Key 'Address_2' has 0 values.
Key 'City' has 0 values.
Key 'Zip' has 0 values.
Key 'Cntry' has 0 values.
Key 'Phone' has 0 values.
Key 'Fax' has 0 values.
Key 'Website' has 0 values.
Key 'Email' has 0 values.
Key 'RegulationType' has 0 values.
Key 'RegulationTypeCode' has 0 values.
Key 'RegulationDate' has 0 values.
Key 'CancellationDate' has 0 values.
Key 'RegCtry' has 0 values.
Key 'RegCode' has 0 values.
Key 'ListCode' has 0 values.
Key 'ListLanguage' has 0 values.
Key 'ListValidityDate' has 0 values.
Key 'ListName' has

In [23]:
### if control room show without remote attribute, than add that one (but too risky)
%pip uninstall -y selenium
%pip install "selenium==4.23.1"

Found existing installation: selenium 4.38.0
Uninstalling selenium-4.38.0:
  Successfully uninstalled selenium-4.38.0
Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/9.4 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.4 MB 26.9 MB/s eta 0:00:01
   ---------------------------------------- 9.4/9.4 MB 34.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seleniumbase 4.44.0 requires selenium==4.38.0; python_version >= "3.10", but you have selenium 4.23.1 which is incompatible.


In [13]:
df = df.drop_duplicates()

In [14]:
df.to_excel('TOTAL_LIST_VER9.xlsx')


In [22]:
soup.find_all('vaadin-vertical-layout')[-1].find('vaadin-grid')

<vaadin-grid all-rows-visible="" style="touch-action: none;" theme="column-borders compact row-stripes"><vaadin-grid-column><template><flow-component-renderer appid="ConsultaDinamica"></flow-component-renderer></template><template class="header"><flow-component-renderer appid="ConsultaDinamica" nodeid="59"></flow-component-renderer></template></vaadin-grid-column><vaadin-grid-column><template><flow-component-renderer appid="ConsultaDinamica"></flow-component-renderer></template><template class="header"><flow-component-renderer appid="ConsultaDinamica" nodeid="56"></flow-component-renderer></template></vaadin-grid-column><vaadin-grid-column><template><flow-component-renderer appid="ConsultaDinamica"></flow-component-renderer></template><template class="header"><flow-component-renderer appid="ConsultaDinamica" nodeid="53"></flow-component-renderer></template></vaadin-grid-column><vaadin-grid-column><template><flow-component-renderer appid="ConsultaDinamica"></flow-component-renderer></te